# PY400 — Reading, Writing, and Manipulating Files

Real data lives in files — CSV, Excel, JSON, Parquet, databases. Pandas makes reading and writing them almost trivial.

## Topics Covered
1. Reading CSV, Excel, JSON, Parquet
2. Writing / exporting data
3. Merging / Joining DataFrames (SQL JOINs)
4. Excel automation (multi-sheet, formatting)
5. Key packages for file I/O

## Key Packages
| Package | Purpose |
|---|---|
| `pandas` | Read/write CSV, Excel, JSON, Parquet, SQL, HTML |
| `openpyxl` | Read/write `.xlsx` Excel files (Pandas uses it under the hood) |
| `xlsxwriter` | Write Excel with advanced formatting, charts, formulas |
| `pyarrow` | Fast Parquet/Arrow I/O (columnar format, great for big data) |
| `sqlalchemy` | Connect Pandas to SQL databases |
| `json` | Built-in Python JSON module |

> **Perspective:** CSV is the universal interchange format — every tool reads it. Parquet is the performance format — 10x smaller and 100x faster to query than CSV for large datasets. Excel is the business format — non-technical stakeholders expect it. Know all three.

---
## 1. Reading Files

In [ ]:
## Create sample data to demonstrate read/write

import pandas as pd
import numpy as np
import json

# Create a sample DataFrame
df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'dept':   ['IT', 'HR', 'IT', 'Finance', 'IT'],
    'salary': [70000, 55000, 85000, 60000, 90000],
    'joined': pd.to_datetime(['2020-01-15', '2019-06-20', '2021-03-10', '2020-11-05', '2018-08-22'])
})

print(df)
print(f"\nTypes:\n{df.dtypes}")

In [ ]:
## Writing files (so we can read them back)

# CSV — universal text format
df.to_csv('sample.csv', index=False)
print("Wrote sample.csv")

# Excel — requires openpyxl
df.to_excel('sample.xlsx', index=False, sheet_name='Employees')
print("Wrote sample.xlsx")

# JSON
df.to_json('sample.json', orient='records', indent=2, date_format='iso')
print("Wrote sample.json")

# Parquet — columnar binary format (fast, compact)
df.to_parquet('sample.parquet', index=False)
print("Wrote sample.parquet")

In [ ]:
## Reading files back

# CSV
df_csv = pd.read_csv('sample.csv', parse_dates=['joined'])
print("=== From CSV ===")
print(df_csv.dtypes)

# Excel
df_excel = pd.read_excel('sample.xlsx', sheet_name='Employees')
print("\n=== From Excel ===")
print(df_excel.head())

# JSON
df_json = pd.read_json('sample.json')
print("\n=== From JSON ===")
print(df_json.head())

# Parquet
df_parquet = pd.read_parquet('sample.parquet')
print("\n=== From Parquet ===")
print(df_parquet.head())

# Key read_csv parameters:
# sep=',' (delimiter), header=0 (row for column names), 
# usecols=['col1','col2'] (load specific columns only),
# nrows=1000 (sample large files), dtype={'col': str} (force types)

---
## 2. Merging / Joining DataFrames (SQL JOINs)

In [ ]:
## SQL-style JOINs with pd.merge()

employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'name':   ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'dept_id': [10, 20, 10, 30, 20]
})

departments = pd.DataFrame({
    'dept_id':   [10, 20, 40],
    'dept_name': ['IT', 'HR', 'Marketing']
})

# INNER JOIN — only matching rows
inner = pd.merge(employees, departments, on='dept_id', how='inner')
print("=== INNER JOIN ===")
print(inner)

# LEFT JOIN — all employees, matching depts (Diana's dept_id=30 has no match)
left = pd.merge(employees, departments, on='dept_id', how='left')
print("\n=== LEFT JOIN ===")
print(left)

# RIGHT JOIN — all departments, matching employees (Marketing has no match)
right = pd.merge(employees, departments, on='dept_id', how='right')
print("\n=== RIGHT JOIN ===")
print(right)

# OUTER JOIN — all rows from both
outer = pd.merge(employees, departments, on='dept_id', how='outer')
print("\n=== OUTER (FULL) JOIN ===")
print(outer)

In [ ]:
## Concatenation (SQL UNION / stacking)

q1 = pd.DataFrame({'month': ['Jan', 'Feb', 'Mar'], 'sales': [100, 120, 115]})
q2 = pd.DataFrame({'month': ['Apr', 'May', 'Jun'], 'sales': [130, 145, 160]})

# Vertical stack (UNION ALL)
yearly = pd.concat([q1, q2], ignore_index=True)
print("=== Vertical concat (UNION) ===")
print(yearly)

# Horizontal stack (add columns side by side)
extra = pd.DataFrame({'target': [110, 125, 120, 135, 150, 165]})
combined = pd.concat([yearly, extra], axis=1)
print("\n=== Horizontal concat ===")
print(combined)

---
## 3. Excel Automation — Multi-sheet, Formatting

In [ ]:
## Writing multiple sheets to one Excel file

with pd.ExcelWriter('report.xlsx', engine='openpyxl') as writer:
    employees.to_excel(writer, sheet_name='Employees', index=False)
    departments.to_excel(writer, sheet_name='Departments', index=False)
    yearly.to_excel(writer, sheet_name='Sales', index=False)

print("Wrote report.xlsx with 3 sheets")

# Reading all sheets at once
all_sheets = pd.read_excel('report.xlsx', sheet_name=None)  # None = all sheets
print(f"\nSheets found: {list(all_sheets.keys())}")
for name, sheet_df in all_sheets.items():
    print(f"  {name}: {sheet_df.shape[0]} rows, {sheet_df.shape[1]} cols")

---
## File Format Comparison

| Format | Human Readable | Size | Read Speed | Best For |
|---|---|---|---|---|
| **CSV** | Yes | Large | Slow | Universal interchange |
| **Excel (.xlsx)** | Yes (in Excel) | Medium | Medium | Business stakeholders |
| **JSON** | Yes | Large | Medium | APIs, nested data |
| **Parquet** | No (binary) | Very small | Very fast | Big data, analytics |
| **Feather** | No (binary) | Small | Fastest | Pandas-to-Pandas transfer |
| **SQLite** | No (binary) | Small | Fast | Local database queries |

> **Best Practice:** For data pipelines, use Parquet — it preserves types, compresses well, and reads 10–100x faster than CSV. Reserve CSV for sharing with non-technical stakeholders or systems that only accept text.

In [ ]:
## Cleanup — remove sample files

import os
for f in ['sample.csv', 'sample.xlsx', 'sample.json', 'sample.parquet', 'report.xlsx']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")